In [20]:
import os
from pathlib import Path
from shutil import copy2

In [21]:
from pathlib import Path
import os
from shutil import copy2

datasets = ["dataset1", "dataset2", "dataset3", "dataset4"]
dataset_ids = ["ds1", "ds2", "ds3", "ds4"]

merged_folder = "merged_dataset"
os.makedirs(merged_folder, exist_ok=True)

for ds_path, ds_id in zip(datasets, dataset_ids):
    ds_path = Path(ds_path)

    for class_folder in ds_path.iterdir():
        if class_folder.is_dir():   # FIXED HERE
            target_class = Path(merged_folder) / class_folder.name
            target_class.mkdir(exist_ok=True)

            for img in class_folder.iterdir():
                if img.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".dcm"]:
                    new_name = f"{ds_id}_{img.name}"
                    copy2(img, target_class / new_name)

print("Merge complete! All datasets are now in", merged_folder)

Merge complete! All datasets are now in merged_dataset


In [22]:
from pathlib import Path
from shutil import copy2

source_dir = Path("merged_dataset")
target_dir = Path("merged_3class")

CLASS_MAP = {
    "A": "M",
    "B": "B",
    "C": "M",
    "L": "M",
    "M": "M",
    "N": "N"
}

for cls in ["B", "M", "N"]:
    (target_dir / cls).mkdir(parents=True, exist_ok=True)

for class_folder in source_dir.iterdir():
    if not class_folder.is_dir():
        continue

    original_class = class_folder.name
    if original_class not in CLASS_MAP:
        continue

    new_class = CLASS_MAP[original_class]

    for img in class_folder.iterdir():
        if img.suffix.lower() in [
            ".png", ".jpg", ".jpeg", ".bmp",
            ".tif", ".tiff", ".dcm"
        ]:
            # unique filename
            new_name = f"{original_class}_{img.parent.name}_{img.name}"
            copy2(img, target_dir / new_class / new_name)

print("Class remapping complete without overwrites!")


Class remapping complete without overwrites!


Hashing


In [23]:
import hashlib

In [24]:
def compute_md5(file_path, block_size=65536):
    md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(block_size), b""):
            md5.update(chunk)
    return md5.hexdigest()


In [25]:
DATASET_DIR = Path("merged_3class")
hash_to_file = {}      # md5 -> first seen file
duplicate_files = []  # files to be removed

valid_ext = {
    ".png", ".jpg", ".jpeg", ".bmp",
    ".tif", ".tiff", ".dcm"
}

for class_dir in DATASET_DIR.iterdir():
    if not class_dir.is_dir():
        continue

    for img_path in class_dir.iterdir():
        if img_path.suffix.lower() not in valid_ext:
            continue

        file_hash = compute_md5(img_path)

        if file_hash in hash_to_file:
            duplicate_files.append(img_path)
        else:
            hash_to_file[file_hash] = img_path


In [26]:
print(f"Total duplicate images detected: {len(duplicate_files)}")

print("\nSample duplicates:")
for dup in duplicate_files[:10]:
    print(dup)


Total duplicate images detected: 2385

Sample duplicates:
merged_3class\B\B_B_ds1_100-001 (258).dcm.png
merged_3class\B\B_B_ds1_100-001 (259).dcm.png
merged_3class\B\B_B_ds1_100-001 (260).dcm.png
merged_3class\B\B_B_ds1_100-001 (261).dcm.png
merged_3class\B\B_B_ds1_100-001 (262).dcm.png
merged_3class\B\B_B_ds1_100-001 (263).dcm.png
merged_3class\B\B_B_ds1_100-001 (264).dcm.png
merged_3class\B\B_B_ds1_100-001 (265).dcm.png
merged_3class\B\B_B_ds1_100-001 (266).dcm.png
merged_3class\B\B_B_ds1_100-001 (267).dcm.png
